# Lab 2.3 - Quantization: Post-Training vs Quantization-Aware Training (IMU gesture model)

This notebook has `# TODO` markers (TODO 1-6). Work through them in order.

Copy your `imu_dataset.npz` and `magic_wand_cnn.h5` from Lab 2.2 into this
same folder before you start.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(10)
np.random.seed(10)

print("TensorFlow version:", tf.__version__)

## Step 0 - Load data and the Lab 2.2 model

Load the `.npz` file you generated with `prepare_dataset()` in Lab 2.2 and
add back the Conv2D channel dimension, then load your trained model. This
part is provided for you.

Note: loaded with `tf.keras.models.load_model` (not `tf_keras.models.load_model`)
- unlike the LCD track, Lab 2.2's `src/cnn2d_model.py` built this model with
plain `tf.keras` (Keras 3), not the legacy `tf_keras` package. We only pull
in `tf_keras` later, in Step 3, where `quantize_model()` specifically
requires it.

In [ ]:
data = np.load("imu_dataset.npz", allow_pickle=False)
class_names = [str(name) for name in data["class_names"]]

X_train = data["X_train"].astype("float32")[..., np.newaxis]
y_train = data["y_train"].astype("int64")
X_test  = data["X_test"].astype("float32")[..., np.newaxis]
y_test  = data["y_test"].astype("int64")

model = tf.keras.models.load_model("magic_wand_cnn.h5")
model.summary()

## Step 1 - Baseline

**TODO 1:** Record the float32 model's size on disk (`os.path.getsize`, in KB)
and its test accuracy (`model.evaluate`). Also measure latency: the average
time for a single-gesture forward pass, calling the model directly
(`model(x, training=False)`) rather than `.predict()`, over up to 100 test
gestures (use `min(100, len(X_test))` - your collected test set may be
smaller than that).

In [ ]:
# TODO 1: size (KB), accuracy, and per-gesture latency (ms) of the baseline model
baseline_size_kb = None
baseline_acc = None

n_latency = min(100, len(X_test))
start = time.time()
for i in range(n_latency):
    pass  # TODO: call the model on a single gesture, X_test[i:i+1]
baseline_latency_ms = None

print(f"Baseline float32 -- size: {baseline_size_kb:.1f} KB, "
      f"accuracy: {baseline_acc:.4f}, latency: {baseline_latency_ms:.3f} ms/gesture")

## Step 2 - Post-Training Quantization (PTQ)

`tf.lite.TFLiteConverter` needs a **representative dataset** - a small sample
of real inputs - to figure out the right scale/zero-point for quantizing
activations. A few dozen to a couple hundred training gestures is plenty
(provided for you below; it already caps at however many training examples
you actually collected).

**TODO 2:** Configure the converter for full-integer quantization:
- `optimizations = [tf.lite.Optimize.DEFAULT]`
- `representative_dataset = representative_dataset`
- `target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]`
- `inference_input_type = tf.uint8`
- `inference_output_type = tf.uint8`

The last two matter for Lab 2.4 - `pico_sw/imu`'s firmware
(`model.cpp`/`main.cpp`) reads `input->data.uint8`/`output->data.uint8`, so
the microcontroller runtime expects **uint8** tensors in and out here (not
int8 like the LCD track) - weights/activations are still quantized to int8
internally via `TFLITE_BUILTINS_INT8`; only the input/output tensor dtype
changes.

In [ ]:
def representative_dataset():
    for i in range(min(200, len(X_train))):
        yield [X_train[i:i+1]]

# TODO 2: build and configure the converter, then convert and save ptq_model.tflite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_ptq = None

with open("ptq_model.tflite", "wb") as f:
    f.write(tflite_ptq)

ptq_size_kb = len(tflite_ptq) / 1024
print(f"PTQ model size: {ptq_size_kb:.1f} KB "
      f"({baseline_size_kb / ptq_size_kb:.1f}x smaller than float32)")

**Evaluating a TFLite model is different from evaluating a Keras model** -
you drive the `Interpreter` directly, and since the input tensor is now
`uint8`, you have to quantize each gesture yourself using the input tensor's
`(scale, zero_point)` before feeding it in:

```
quantized_value = round(float_value / scale + zero_point)  # clipped to [0, 255]
```

The output is uint8 too, but since quantization is a monotonic
(order-preserving) transform, `argmax` on the raw uint8 output gives the same
predicted class as `argmax` on the dequantized version - no need to
dequantize just to get a prediction.

**TODO 3:** Complete `evaluate_tflite()`: quantize each input, run it through
the interpreter, and check whether the predicted class matches the true
label. Also time 100 invocations (or fewer, if the dataset is smaller) for a
latency figure, the same way as Step 1.

In [ ]:
def evaluate_tflite(tflite_bytes, x, y, n_latency=100):
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    scale, zero_point = inp["quantization"]

    def quantize(gesture):
        # TODO 3a: quantize a float gesture using (scale, zero_point)
        return None

    # TODO 3b: loop over all of x/y, run inference, count correct predictions
    correct = 0
    accuracy = None

    # TODO 3c: time min(n_latency, len(x)) invocations (same pattern as Step 1)
    latency_ms = None

    return accuracy, latency_ms, inp["dtype"], out["dtype"]

ptq_acc, ptq_latency_ms, ptq_in_dtype, ptq_out_dtype = evaluate_tflite(tflite_ptq, X_test, y_test)
print(f"PTQ -- accuracy: {ptq_acc:.4f}, latency: {ptq_latency_ms:.3f} ms/gesture, "
      f"input dtype: {ptq_in_dtype}, output dtype: {ptq_out_dtype}")

## Step 3 - Quantization-Aware Training (QAT)

`tensorflow_model_optimization` depends on the `tf_keras` package (legacy
Keras 2) rather than the Keras 3 that's `tf.keras` by default in recent
TensorFlow - `quantize_model()` specifically requires a `tf_keras`
`Sequential`/`Functional` model.

Unlike the LCD track, the IMU model in Lab 2.2 (`src/cnn2d_model.py`) was
built with plain `tf.keras` (Keras 3), not `tf_keras` - so `model`, loaded
in Step 0, isn't directly compatible with `quantize_model()`, and `tf_keras`
can't load a Keras-3-saved `.h5` file either (their serialization formats
differ). `clone_as_tf_keras()` below works around this: it rebuilds the
same layer stack (Conv2D / MaxPooling2D / Flatten / Dropout / Dense) using
`tf_keras` layers and copies the trained weights across. It's provided for
you - reconstructing Lab 2.2's architecture isn't the point of this lab.

**TODO 4:** Import `tensorflow_model_optimization as tfmot`, wrap the
`tf_keras_model` twin built below with `tfmot.quantization.keras.quantize_model()`,
compile it (same loss/metric as Step 1), and fine-tune for 2 epochs.

In [ ]:
import tf_keras


def _activation_name(layer):
    activation = layer.get_config().get("activation", "linear")
    return activation if isinstance(activation, str) else getattr(activation, "__name__", "linear")


def clone_as_tf_keras(keras3_model):
    """Rebuild a tf.keras (Keras 3) Sequential model as an equivalent
    tf_keras (legacy Keras 2) model, then copy over the trained weights."""
    twin_layers = [tf_keras.layers.Input(shape=keras3_model.input_shape[1:])]

    for layer in keras3_model.layers:
        name = layer.__class__.__name__
        if name == "Conv2D":
            twin_layers.append(tf_keras.layers.Conv2D(
                filters=layer.filters, kernel_size=layer.kernel_size,
                strides=layer.strides, padding=layer.padding,
                activation=_activation_name(layer)))
        elif name == "MaxPooling2D":
            twin_layers.append(tf_keras.layers.MaxPooling2D(
                pool_size=layer.pool_size, strides=layer.strides,
                padding=layer.padding))
        elif name == "Flatten":
            twin_layers.append(tf_keras.layers.Flatten())
        elif name == "Dropout":
            twin_layers.append(tf_keras.layers.Dropout(rate=layer.rate))
        elif name == "Dense":
            twin_layers.append(tf_keras.layers.Dense(
                units=layer.units, activation=_activation_name(layer)))
        else:
            raise ValueError(f"Unsupported layer type for cloning: {name}")

    twin = tf_keras.Sequential(twin_layers, name=keras3_model.name)
    twin.set_weights(keras3_model.get_weights())
    return twin


tf_keras_model = clone_as_tf_keras(model)
tf_keras_model.summary()

In [ ]:
# TODO 4: import tfmot, wrap `tf_keras_model`, compile, and fine-tune
qat_model = None

**TODO 5:** Convert `qat_model` to a fully-quantized uint8 TFLite model -
same converter settings as Step 2 - and evaluate it with your
`evaluate_tflite()` from Step 2.

In [ ]:
# TODO 5: convert qat_model to uint8 TFLite, save as qat_model.tflite, then evaluate
tflite_qat = None

with open("qat_model.tflite", "wb") as f:
    f.write(tflite_qat)

qat_size_kb = len(tflite_qat) / 1024
qat_acc, qat_latency_ms, qat_in_dtype, qat_out_dtype = evaluate_tflite(tflite_qat, X_test, y_test)
print(f"QAT -- size: {qat_size_kb:.1f} KB, accuracy: {qat_acc:.4f}, "
      f"latency: {qat_latency_ms:.3f} ms/gesture, "
      f"input dtype: {qat_in_dtype}, output dtype: {qat_out_dtype}")

## Step 4 - Compare

Check that both quantized models have `uint8` input/output tensors printed
above - that's the requirement `pico_sw/imu`'s microcontroller runtime
needs.

**TODO 6:** Build a comparison table (a small `pandas.DataFrame` is easiest)
with one row per variant (float32 baseline, PTQ uint8, QAT uint8) and
columns for size (KB), accuracy, and latency (ms). Then plot size and
accuracy side by side as bar charts.

In [ ]:
# TODO 6: build the comparison DataFrame
comparison = None
comparison

In [ ]:
# TODO 6 (continued): bar charts for size and accuracy, side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

plt.tight_layout()
plt.savefig("quantization_comparison.png", dpi=110)
plt.show()

## Discussion

In one paragraph: did QAT actually beat PTQ here? By how much, and was the
extra training time (and the `tf_keras` conversion step it required) worth
it for your use case? Also note how much the size of your collected gesture
dataset (versus MNIST's ~70,000 images) affected your confidence in these
numbers - a handful of test gestures per class makes the accuracy figures
much noisier than MNIST's.